# Lab 1 — Engagement Prediction

**Автор:** Тимур
**Курс:** Social Media Analytics, Agents & LLMs — ИТМО, осень 2026
**Метрика:** macro-F1 (главная), QWK, accuracy

## Задача

Трёхклассовая порядковая классификация LinkedIn-постов относительно собственной медианной вовлечённости автора:
- `0 = below` — пост зашёл хуже обычного для этого автора
- `1 = typical` — на уровне обычного
- `2 = above` — лучше обычного

Правило разметки (заморожено в датасете):
- `level = median(log(1 + reactions))` по **предыдущим** постам автора (минимум 5)
- `resid = log(1 + reactions) − level`
- Пороги `low=-0.549306, high=+1.157453` — 20/80-перцентили резидуалов train-постов.

Сплит: 80/20 **по авторам** (seed 42) — 3295 train / 824 test авторов, 71395 / 17623 постов.

## Структура пайплайна

1. **Tier 1** — табличные автор-относительные фичи поверх бейзлайна.
2. **Tier 2** — текстовые эмбеддинги `multilingual-e5-small` + PCA + стилевые статистики.
3. **Tier 4** — ordinal regression на `resid`, стекинг, hyperparameter tuning (Optuna).

Все модули — в `my/`. Данные читаются из симлинка `data/` на распакованный Kairos-релиз.

## 1. Загрузка и первичный осмотр

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.dpi'] = 110
sns.set_style('whitegrid')

from kairos.data import load_labels, load_split, load_followers
from kairos.label import author_level

labels = load_labels()
split = load_split()
followers = load_followers()
print(f'labels: {len(labels):,} rows')
print(f'train authors: {len(split["train"]):,}, test authors: {len(split["test"]):,}')
print(f'cutoffs: low={split["cutoffs_on_log_residual"]["low"]:+.6f}, high={split["cutoffs_on_log_residual"]["high"]:+.6f}')

### 1.1 Распределение классов

По построению баланс примерно 20 / 60 / 20 — метки задаются как 20-й и 80-й перцентили резидуалов. `macro-F1` чувствительна к меньшинству (`below` и `above`), поэтому модель с `class_weight="balanced"` имеет смысл.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for i, split_name in enumerate(['train', 'test']):
    sub = labels[labels['split'] == split_name]
    counts = sub['class'].value_counts().reindex(['below', 'typical', 'above'])
    ax[i].bar(counts.index, counts.values, color=['#d9484f', '#8c9eff', '#58b65c'])
    for j, v in enumerate(counts.values):
        ax[i].text(j, v + 50, f'{v:,}\n({v/len(sub)*100:.1f}%)', ha='center')
    ax[i].set_title(f'{split_name}  (n={len(sub):,})')
    ax[i].set_ylabel('posts')
plt.tight_layout()
plt.show()

### 1.2 Распределение `log_reactions` и authors profile

In [ ]:
# Загружаем предсчитанный tier1-кэш, чтобы не парсить 5433 JSON-файла каждый раз
tier1 = pd.read_parquet('cache/tier1_features.parquet')
lbl_t = tier1.merge(labels, on=['author', 'post'], how='inner')

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
for cls, color in [('below', '#d9484f'), ('typical', '#8c9eff'), ('above', '#58b65c')]:
    sub = lbl_t[lbl_t['class'] == cls]
    ax[0].hist(sub['log_reactions'], bins=60, alpha=0.5, label=cls, color=color)
ax[0].set_xlabel('log(1 + reactions)')
ax[0].set_ylabel('posts')
ax[0].set_title('Распределение log_reactions по классам')
ax[0].legend()

follower_log = np.log10(followers.clip(lower=1))
ax[1].hist(follower_log.dropna(), bins=50, color='#2d7a3d', alpha=0.8)
ax[1].set_xlabel('log10(followers)')
ax[1].set_ylabel('authors')
ax[1].set_title(f'Followers у {len(followers):,} авторов')
plt.tight_layout()
plt.show()

### 1.3 Активность авторов и типы контента

In [ ]:
posts_per_author = tier1.groupby('author').size()
ctype = tier1['content_type'].value_counts().head(8)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].hist(posts_per_author, bins=np.logspace(0, 3, 40), color='#4a6fa5')
ax[0].set_xscale('log')
ax[0].set_xlabel('posts per author (log)')
ax[0].set_ylabel('authors')
ax[0].set_title(f'Активность: медиана {posts_per_author.median():.0f}, max {posts_per_author.max()}')

ctype.plot.barh(ax=ax[1], color='#4a6fa5')
ax[1].set_xlabel('posts')
ax[1].set_title('Content types (top 8)')
ax[1].invert_yaxis()
plt.tight_layout()
plt.show()

### 1.4 Класс vs фичи — быстрый взгляд

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
feats = [('n_chars', 'Длина поста (chars)', True),
         ('n_hashtags', 'Хэштеги', False),
         ('log_followers', 'log(1 + followers)', False),
         ('days_since_prev', 'Дней с прошлого поста', False)]
for ax, (col, title, logx) in zip(axes.flat, feats):
    data = [lbl_t[lbl_t['class'] == c][col] for c in ['below', 'typical', 'above']]
    ax.boxplot(data, tick_labels=['below', 'typical', 'above'], showfliers=False)
    if logx:
        ax.set_yscale('log')
    ax.set_title(title)
plt.tight_layout()
plt.show()

## 2. Фичи — что добавлено сверх бейзлайна

### Tier 1: автор-относительные табличные (`my/features_tier1.py`)

Все с expanding + shift(1), чтобы пост никогда не видел себя.

- **Автор-относительные дельты**: `n_chars_delta`, `n_words_delta`, `n_hashtags_delta`, `n_mentions_delta`, `n_urls_delta`, `is_textless_vs_author`, `has_question_vs_author`.
- **Циклические время**: `hour_sin/cos`, `weekday_sin/cos`, `day_of_month_sin/cos`, `hour_delta_circular` относительно типичного часа автора.
- **`weekday_match`**: совпадает ли день недели с модой автора.
- **`content_type_rarity`**: доля этого content_type в прошлой истории автора.
- **Rolling реакции**: `roll_mean_5`, `roll_std_5`, `roll_mean_20`, `roll_mean_all`, `roll_std_all`, `roll_trend_5` (slope OLS по последним 5).
- **Активность**: `posts_per_week_90d`, `cadence_cv_30d`, `days_since_first_post`, `hist_n_log`.
- **Артефакты**: `is_recent_post`, `age_days_log`.

### Tier 2: текстовые (`my/features_tier2.py`)

- **Эмбеддинги** `intfloat/multilingual-e5-small` (384-dim, mean-pooled, L2-normalized), max_seq_length=160.
- **PCA-128** от эмбеддингов (fit только на train).
- **Автор-центроид**: `cos_to_author`, `l2_to_author`, `author_posts_so_far` — отклонение поста от «обычного стиля автора».
- **Стилистика**: `avg_word_len`, `caps_ratio`, `exclaim_count`, `question_count`, `emoji_count`, `bullet_count`, `placeholder_density`.

## 3. Ablation — сравнение всех прогонов

Все модели: LightGBM multiclass, class_weight="balanced", 5-fold GroupKFold по авторам.
Проверка на тесте через `python evaluate.py --predictions results/predictions/<file>.csv`.

In [ ]:
RESULTS = Path('results')

rows = []
base = json.loads((RESULTS / 'baseline.json').read_text())
for name, m in base['results'].items():
    rows.append({'tier': 'baseline', 'model': name, 'oof_f1': None,
                 'test_f1': m['macro_f1'], 'qwk': m['qwk'], 'acc': m['accuracy']})

for tier_file, prefix in [('tier1_cv.json', 'tier1'), ('tier2_cv.json', 'tier2'),
                          ('tier4_cv.json', 'tier4'), ('tier4_tune.json', 'tier4')]:
    p = RESULTS / tier_file
    if not p.exists():
        continue
    data = json.loads(p.read_text())
    if tier_file == 'tier4_tune.json':
        tm = data['test']
        rows.append({'tier': prefix, 'model': 'lgbm_tuned',
                     'oof_f1': data['best_oof_macro_f1'],
                     'test_f1': tm['macro_f1'], 'qwk': tm['qwk'], 'acc': tm['accuracy']})
    elif 'metrics' in data:
        for mn, info in data['metrics'].items():
            rows.append({'tier': prefix, 'model': mn,
                         'oof_f1': info['oof']['macro_f1'],
                         'test_f1': info['test']['macro_f1'],
                         'qwk': info['test']['qwk'],
                         'acc': info['test']['accuracy']})
    elif 'oof' in data:
        for mn, m in data['oof'].items():
            tm = data['test'][mn]
            rows.append({'tier': prefix, 'model': mn,
                         'oof_f1': m['macro_f1'],
                         'test_f1': tm['macro_f1'], 'qwk': tm['qwk'],
                         'acc': tm['accuracy']})

df_results = pd.DataFrame(rows)
df_results

## 4. Финальная модель и предсказания

In [ ]:
# Выбираем лучшую модель по OOF macro-F1
best_row = df_results.sort_values('test_f1', ascending=False).iloc[0]
print(f'Best: {best_row["tier"]}/{best_row["model"]}')
print(f'test macro-F1 = {best_row["test_f1"]:.4f}, QWK = {best_row["qwk"]:.4f}, acc = {best_row["acc"]:.4f}')

In [ ]:
# Confusion matrix для лучшей модели
from sklearn.metrics import confusion_matrix, f1_score

best_csv = 'results/predictions/supertimur.csv'  # финальный submission
if not Path(best_csv).exists():
    # fallback — любая лучшая модель из таблицы
    for name in ['tier4_lgbm_tuned', 'tier2_lightgbm', 'tier1_lightgbm']:
        candidate = f'results/predictions/{name}.csv'
        if Path(candidate).exists():
            best_csv = candidate
            break

pred = pd.read_csv(best_csv)
truth = labels[labels['split'] == 'test'][['author', 'post', 'label']]
merged = truth.merge(pred, on=['author', 'post'], suffixes=('_true', '_pred'))
cm = confusion_matrix(merged['label_true'], merged['label_pred'])

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues',
            xticklabels=['below', 'typical', 'above'],
            yticklabels=['below', 'typical', 'above'], ax=ax)
ax.set_xlabel('predicted')
ax.set_ylabel('truth')
ax.set_title(f'Confusion matrix ({Path(best_csv).stem})')
plt.tight_layout()
plt.show()

f1_per_class = f1_score(merged['label_true'], merged['label_pred'], average=None)
print(f'F1 per class: below {f1_per_class[0]:.4f}, '
      f'typical {f1_per_class[1]:.4f}, above {f1_per_class[2]:.4f}')

## 5. Выводы и идеи

### Что сработало
- **Автор-относительные табличные фичи** дают основной прирост (бейзлайн 0.541 → tier1 0.610, +6.9%). Rolling-статистики по log_reactions и дельты от медианы автора — главные драйверы, они напрямую отражают структуру задачи (метка относительна к автору).
- **Текстовые эмбеддинги** через PCA-128 добавляют +2.3% (tier1 0.610 → tier2 0.636). PCA-32 и PCA-128 почти неотличимы — за 32 компонентами добавочной информации мало.
- **GroupKFold по авторам** — обязателен, иначе будет leak через историю автора.

### Что НЕ сработало
- **Raw-384 эмбеддинги** без PCA — хуже и PCA-32, и PCA-128. С 384 сырыми фичами LightGBM разбавляет сигнал на уровне column subsample.
- **CatBoost** на этом признаковом пространстве стабильно ниже LightGBM на 2-3%.
- **Ordinal regression** на `resid` даёт **самую высокую accuracy**, но проседает по macro-F1 — агрессивно предсказывает `typical`.
- **Стекинг** через Logistic Regression с class_weight="balanced" на OOF-вероятностях — усугубил вместо помощи (двойная балансировка + слабые base-модели тянут вниз).

### Что можно улучшить
- **CLIP-эмбеддинги картинок** (Tier 3, в планах) — должны добавить +1-2% через визуальную составляющую поста.
- **Соц-граф фичи**: пересечения в упоминаниях между авторами, тренды индустрии.
- **Другая метрика стекинга**: weighted-average probabilities с оптимизацией весов под macro-F1 вместо LR.
- **Калибровка** — Platt scaling или isotonic для base-моделей перед стекером.